In [ ]:
NISAR_L2_UR_GOFF_031_141_D_084_032_4000_SH_20260927T235307_20260927T235336_20261009T235308_20261009T235336_P05023_F_F_J_001.h5 
NISAR_L2_UR_GOFF_031_141_D_085_032_2000_SH_20260927T235335_20260927T235348_20261009T235335_20261009T235349_P05023_F_P_J_001.h5 
NISAR_L2_UR_GOFF_031_141_D_085_032_4000_SH_20260927T235335_20260927T235344_20261009T235335_20261009T235345_P05023_F_P_J_001.h5 
NISAR_L2_UR_GUNW_031_141_D_085_032_2000_SH_20260927T235335_20260927T235348_20261009T235335_20261009T235349_P05023_F_P_J_001.h5 
NISAR_L2_UR_GUNW_031_141_D_085_032_4000_SH_20260927T235335_20260927T235344_20261009T235335_20261009T235345_P05023_F_P_J_001.h5

## NISAR tif extraction

In [ ]:
### GUNW

In [ ]:
%%bash
set -uo pipefail

INPUT_DIR="/Users/chanagan/Downloads/Panama2026/NISAR_ASF"
OUTPUT_DIR="${INPUT_DIR}/nisar_gunw_tifs"
TARGET_CRS="EPSG:32617"
AOI="/Users/chanagan/Downloads/Panama2026/GIS/BigAOIPanama2026.shp"
POLARIZATION="HH"

mkdir -p "$OUTPUT_DIR"

if [[ ! -f "$AOI" ]]; then
    echo "ERROR: AOI not found: $AOI"
    exit 1
fi

shopt -s nullglob
files=("$INPUT_DIR"/*GUNW*.h5)

if (( ${#files[@]} == 0 )); then
    echo "ERROR: No *GUNW*.h5 files found in $INPUT_DIR"
    exit 1
fi

echo "Found ${#files[@]} GUNW files."

success=0
failed=0

# Export one real-valued HDF5 dataset to a clipped GeoTIFF
export_raster() {
    local h5="$1"
    local dataset="$2"
    local output="$3"

    local input_dataset="HDF5:\"${h5}\":${dataset}"

    echo "  Exporting: $dataset"

    if ! gdalinfo "$input_dataset" >/dev/null 2>&1; then
        echo "  ERROR: Cannot open dataset: $dataset"
        ((failed+=1))
        return
    fi

    if gdalwarp \
        -t_srs "$TARGET_CRS" \
        -overwrite \
        -cutline "$AOI" \
        -crop_to_cutline \
        -r near \
        -of GTiff \
        -ot Float32 \
        -dstnodata -9999 \
        -co COMPRESS=LZW \
        -co ZLEVEL=9 \
        -co BIGTIFF=YES \
        "$input_dataset" \
        "$output"; then

        echo "  SUCCESS: $output"
        ((success+=1))
    else
        echo "  ERROR: gdalwarp failed"
        rm -f "$output"
        ((failed+=1))
    fi
}

for h5 in "${files[@]}"; do
    filename=$(basename "$h5" .h5)

    echo
    echo "Processing: $filename"

    base="//science/LSAR/GUNW/grids/frequencyA"
    pol="$POLARIZATION"

    # 1. Unwrapped phase (radians)
    export_raster "$h5" \
        "${base}/unwrappedInterferogram/${pol}/unwrappedPhase" \
        "${OUTPUT_DIR}/${filename}_unwrappedPhase.tif"

    # 2. Slant-range pixel offset
    export_raster "$h5" \
        "${base}/pixelOffsets/${pol}/slantRangeOffset" \
        "${OUTPUT_DIR}/${filename}_slantRangeOffset.tif"

    # 3. Along-track pixel offset
    export_raster "$h5" \
        "${base}/pixelOffsets/${pol}/alongTrackOffset" \
        "${OUTPUT_DIR}/${filename}_alongTrackOffset.tif"

    # 4. Wrapped interferometric phase (radians)
    # The source is complex-valued, so extract PHASE before warping.
    wrapped_path="${base}/wrappedInterferogram/${pol}/wrappedInterferogram"
    wrapped_source="HDF5:\"${h5}\":${wrapped_path}"
    phase_source="DERIVED_SUBDATASET:PHASE:${wrapped_source}"
    phase_tmp="${OUTPUT_DIR}/${filename}_wrappedPhase_tmp.tif"
    phase_out="${OUTPUT_DIR}/${filename}_wrappedPhase.tif"

    echo "  Extracting wrapped interferometric phase"

    if gdalinfo "$phase_source" >/dev/null 2>&1; then
        if gdalwarp \
            -t_srs "$TARGET_CRS" \
            -overwrite \
            -cutline "$AOI" \
            -crop_to_cutline \
            -r near \
            -of GTiff \
            -ot Float32 \
            -dstnodata -9999 \
            -co COMPRESS=LZW \
            -co ZLEVEL=9 \
            -co BIGTIFF=YES \
            "$phase_source" \
            "$phase_out"; then

            echo "  SUCCESS: $phase_out"
            ((success+=1))
        else
            echo "  ERROR: Wrapped-phase warp failed"
            rm -f "$phase_out"
            ((failed+=1))
        fi
    else
        echo "  WARNING: GDAL cannot open the derived phase source."
        echo "  Inspect the complex dataset with gdalinfo."
        ((failed+=1))
    fi

done

echo
echo "========================================"
echo "Finished processing GUNW files."
echo "Successful outputs: $success"
echo "Failed datasets:    $failed"
echo "Output directory:   $OUTPUT_DIR"
echo "========================================"


### GOFF

In [ ]:

%%bash
set -uo pipefail

INPUT_DIR="/Users/chanagan/Downloads/Panama2026/NISAR_ASF"
OUTPUT_DIR="${INPUT_DIR}/nisar_offsets"
TARGET_CRS="EPSG:32617"
AOI="/Users/chanagan/Downloads/Panama2026/GIS/BigAOIPanama2026.shp"

mkdir -p "$OUTPUT_DIR"

# Check the AOI
if [[ ! -f "$AOI" ]]; then
    echo "ERROR: AOI shapefile not found: $AOI"
    exit 1
fi

# Find GOFF files
shopt -s nullglob
files=("$INPUT_DIR"/*GOFF*.h5)

if (( ${#files[@]} == 0 )); then
    echo "ERROR: No files matching *GOFF*.h5 found in:"
    echo "  $INPUT_DIR"
    echo
    echo "HDF5 files actually present:"
    ls -lh "$INPUT_DIR"/*.h5 2>/dev/null || true
    exit 1
fi

echo "Found ${#files[@]} GOFF file(s)."
echo "Output directory: $OUTPUT_DIR"

success=0
failed=0

for h5 in "${files[@]}"; do
    filename=$(basename "$h5" .h5)
    echo
    echo "Processing: $filename"

    for layer in layer1 layer2 layer3; do
        for component in slantRangeOffset alongTrackOffset; do

            dataset="//science/LSAR/GOFF/grids/frequencyA/pixelOffsets/HH/${layer}/${component}"
            output="${OUTPUT_DIR}/${filename}_${layer}_${component}.tif"

            # Keep the HDF5 subdataset as one shell argument
            input_dataset="HDF5:\"${h5}\":${dataset}"

            echo
            echo "  Dataset: ${layer}/${component}"

            # Confirm GDAL can open the source dataset
            if ! gdalinfo "$input_dataset" >/dev/null 2>&1; then
                echo "  ERROR: Cannot open dataset:"
                echo "  $input_dataset"
                ((failed+=1))
                continue
            fi

            # Reproject and clip
            if gdalwarp \
                -t_srs "$TARGET_CRS" \
                -overwrite \
                -cutline "$AOI" \
                -crop_to_cutline \
                -r near \
                -of GTiff \
                -ot Float32 \
                -dstnodata -9999 \
                -co COMPRESS=LZW \
                -co ZLEVEL=9 \
                -co BIGTIFF=YES \
                "$input_dataset" \
                "$output"; then

                echo "  SUCCESS: $output"
                ((success+=1))
            else
                echo "  ERROR: gdalwarp failed."
                rm -f "$output"
                ((failed+=1))
            fi

        done
    done
done

echo
echo "=================================="
echo "Processing complete."
echo "Successful outputs: $success"
echo "Failed datasets:    $failed"
echo "Output directory:   $OUTPUT_DIR"
echo "=================================="


In [ ]:
gdalwarp -t_srs EPSG:32617 -overwrite -crop_to_cutline -cutline /Users/chanagan/Downloads/Panama2026/GIS/BigAOIPanama2026.shp \
          -tr 80 -80 -r near -of GTiff -ot Float32 -dstnodata -9999 -co COMPRESS=LZW -co ZLEVEL=9 -co BIGTIFF=YES \
          /Users/chanagan/Downloads/Panama2026/NISAR_ASF/nisar_offsets/NISAR_L2_UR_GOFF_031_141_D_084_032_4000_SH_20260927T235307_20260927T235336_20261009T235308_20261009T235336_P05023_F_F_J_001_layer2_slantRangeOffset_masked.tif \
/Users/chanagan/Downloads/Panama2026/NISAR_ASF/nisar_offsets/NISAR_L2_UR_GOFF_031_141_D_085_032_2000_SH_20260927T235335_20260927T235348_20261009T235335_20261009T235349_P05023_F_P_J_001_layer2_slantRangeOffset_masked.tif \
/Users/chanagan/Downloads/Panama2026/NISAR_ASF/nisar_offsets/NISAR_L2_UR_GOFF_031_141_D_084_032_4000_SH_20260927T235307_20260927T235336_20261009T235308_20261009T235336_P05023_F_F_J_001_layerMerge_slantRangeOffset_masked.tif

In [ ]:
# Cloud and water mask

import numpy as np
from scipy.ndimage import binary_dilation
import rasterio


def mask_bright_objects(
    arr,
    percentile=99,
    buffer_pixels=4,
    nodata=-9999
):
    """
    Mask bright objects (e.g., clouds) using a brightness threshold.

    Parameters
    ----------
    arr : ndarray
        Single-band image.
    percentile : float
        Brightness percentile above which pixels are masked.
    buffer_pixels : int
        Buffer around masked pixels.
    nodata : float
        Value used for masking.

    Returns
    -------
    masked : ndarray
    mask : ndarray(bool)
    """

    masked = arr.astype(float).copy()

    valid = np.isfinite(masked)

    thresh = np.percentile(masked[valid], percentile)

    mask = masked >= thresh

    mask = binary_dilation(mask, iterations=buffer_pixels)

    masked[mask] = nodata

    return masked, mask

import numpy as np
import geopandas as gpd
from rasterio.features import rasterize
from scipy.ndimage import binary_dilation, binary_erosion


def mask_ocean(
    arr,
    transform,
    crs,
    landmask_file,
    nodata=np.nan,
    coastline_buffer=0,
    all_touched=False,
):
    """
    Mask ocean using a GSHHG land polygon shapefile.

    Parameters
    ----------
    arr : ndarray
        2D raster.
    transform : affine.Affine
        Raster transform.
    crs : rasterio.crs.CRS
        Raster CRS.
    landmask_file : str
        Path to GSHHG land shapefile
        (e.g. GSHHS_f_L1.shp).
    nodata : float
        Output nodata value.
    coastline_buffer : int
        Buffer (pixels).

        >0 : remove pixels near coastline (recommended)
        <0 : expand land into ocean
         0 : no buffering
    all_touched : bool
        Passed to rasterize(). False is usually best.

    Returns
    -------
    masked : ndarray
    landmask : ndarray(bool)
    """

    # Read GSHHG polygons
    gdf = gpd.read_file(landmask_file)

    # Repair invalid geometries
    gdf["geometry"] = gdf.geometry.buffer(0)

    # Match raster CRS
    if gdf.crs != crs:
        print('Projecting to',crs)
        gdf = gdf.to_crs(crs)

        gdf = gdf[gdf.geometry.bounds.apply(
                    lambda row: np.isfinite(row).all(),
                    axis=1)]

    # Rasterize land polygons
    landmask = rasterize(
        ((geom, 1) for geom in gdf.geometry),
        out_shape=arr.shape,
        transform=transform,
        fill=0,
        all_touched=all_touched,
        dtype=np.uint8,
    ).astype(bool)

    # Optional coastline buffer
    if coastline_buffer > 0:
        landmask = binary_erosion(
            landmask,
            iterations=coastline_buffer,
        )

    elif coastline_buffer < 0:
        landmask = binary_dilation(
            landmask,
            iterations=abs(coastline_buffer),
        )

    # Apply mask
    masked = arr.astype(float).copy()
    masked[~landmask] = nodata

    return masked, landmask

files = ['/Users/chanagan/Downloads/Panama2026/NISAR_ASF/nisar_offsets/NISAR_L2_UR_GOFF_031_141_D_084_032_4000_SH_20260927T235307_20260927T235336_20261009T235308_20261009T235336_P05023_F_F_J_001_layer1_alongTrackOffset.tif', 
'/Users/chanagan/Downloads/Panama2026/NISAR_ASF/nisar_offsets/NISAR_L2_UR_GOFF_031_141_D_084_032_4000_SH_20260927T235307_20260927T235336_20261009T235308_20261009T235336_P05023_F_F_J_001_layer1_slantRangeOffset.tif', 
'/Users/chanagan/Downloads/Panama2026/NISAR_ASF/nisar_offsets/NISAR_L2_UR_GOFF_031_141_D_084_032_4000_SH_20260927T235307_20260927T235336_20261009T235308_20261009T235336_P05023_F_F_J_001_layer2_alongTrackOffset.tif', 
'/Users/chanagan/Downloads/Panama2026/NISAR_ASF/nisar_offsets/NISAR_L2_UR_GOFF_031_141_D_084_032_4000_SH_20260927T235307_20260927T235336_20261009T235308_20261009T235336_P05023_F_F_J_001_layer2_slantRangeOffset.tif', 
'/Users/chanagan/Downloads/Panama2026/NISAR_ASF/nisar_offsets/NISAR_L2_UR_GOFF_031_141_D_085_032_2000_SH_20260927T235335_20260927T235348_20261009T235335_20261009T235349_P05023_F_P_J_001_layer1_alongTrackOffset.tif', 
'/Users/chanagan/Downloads/Panama2026/NISAR_ASF/nisar_offsets/NISAR_L2_UR_GOFF_031_141_D_085_032_2000_SH_20260927T235335_20260927T235348_20261009T235335_20261009T235349_P05023_F_P_J_001_layer1_slantRangeOffset.tif', 
'/Users/chanagan/Downloads/Panama2026/NISAR_ASF/nisar_offsets/NISAR_L2_UR_GOFF_031_141_D_085_032_2000_SH_20260927T235335_20260927T235348_20261009T235335_20261009T235349_P05023_F_P_J_001_layer2_alongTrackOffset.tif', 
'/Users/chanagan/Downloads/Panama2026/NISAR_ASF/nisar_offsets/NISAR_L2_UR_GOFF_031_141_D_085_032_2000_SH_20260927T235335_20260927T235348_20261009T235335_20261009T235349_P05023_F_P_J_001_layer2_slantRangeOffset.tif', 
'/Users/chanagan/Downloads/Panama2026/NISAR_ASF/nisar_offsets/NISAR_L2_UR_GOFF_031_141_D_085_032_4000_SH_20260927T235335_20260927T235344_20261009T235335_20261009T235345_P05023_F_P_J_001_layer1_alongTrackOffset.tif', 
'/Users/chanagan/Downloads/Panama2026/NISAR_ASF/nisar_offsets/NISAR_L2_UR_GOFF_031_141_D_085_032_4000_SH_20260927T235335_20260927T235344_20261009T235335_20261009T235345_P05023_F_P_J_001_layer1_slantRangeOffset.tif', 
'/Users/chanagan/Downloads/Panama2026/NISAR_ASF/nisar_offsets/NISAR_L2_UR_GOFF_031_141_D_085_032_4000_SH_20260927T235335_20260927T235344_20261009T235335_20261009T235345_P05023_F_P_J_001_layer2_alongTrackOffset.tif', 
'/Users/chanagan/Downloads/Panama2026/NISAR_ASF/nisar_offsets/NISAR_L2_UR_GOFF_031_141_D_085_032_4000_SH_20260927T235335_20260927T235344_20261009T235335_20261009T235345_P05023_F_P_J_001_layer2_slantRangeOffset.tif']

landmask_file = '/Users/chanagan/Library/CloudStorage/OneDrive-DOI/Data/MaskFiles/gshhg-shp-2.3.7/GSHHS_shp/f/GSHHS_f_L1.shp'

for file in files:
    with rasterio.open(file) as src:
        arr = src.read(1)
        profile = src.profile
        nodata = src.nodata
        transform = src.transform
        crs = src.crs

        masked, _ = mask_ocean(arr,transform,crs,landmask_file,
                nodata=-9999,coastline_buffer=5)
        
        with rasterio.open(f"{file[:-4]}_masked.tif", "w", **profile) as dst:
            dst.write(masked, 1)

## Extract displacements

In [ ]:
from matplotlib.patches import Rectangle
import sys
sys.path.append('/Users/chanagan/Documents/GitHub/ImageryResources/Functions/')
import TiffTools as tt
from shapely.ops import linemerge
def swath_corners(x0, y0, az_deg, length, width):
    az = az_deg * np.pi/180

    # shift vectors
    u_par = np.array([np.sin(az), np.cos(az)])
    u_perp = np.array([np.cos(az), -np.sin(az)])

    hl = length / 2
    hw = width / 2

    shift = np.array([
        np.dot(np.array([-hl,hw]), np.array([u_perp,u_par])),
        np.dot(np.array([hl,hw]), np.array([u_perp,u_par])),
        np.dot(np.array([hl,-hw]), np.array([u_perp,u_par])),
        np.dot(np.array([-hl,-hw]), np.array([u_perp,u_par]))

    ])

    corners = np.array([
        [x0+shift[0,0],y0+shift[0,1]],
        [x0+shift[1,0],y0+shift[1,1]],
        [x0+shift[2,0],y0+shift[2,1]],
        [x0+shift[3,0],y0+shift[3,1]]
    ])

    return corners

In [ ]:
crs = 'EPSG:32617'

simple_trace = gpd.read_file('/Users/chanagan/Downloads/Panama2026/GIS/ProfileTrace.shp')

swath_dict = {
    'Panama':{
             'swath_width':80,
             'swath_length':10000,
             'resolution':80,
             'disp_file':'/Users/chanagan/Downloads/Panama2026/NISAR_ASF/nisar_offsets/NISAR_L2_UR_GOFF_031_141_D_084_032_4000_SH_20260927T235307_20260927T235336_20261009T235308_20261009T235336_P05023_F_F_J_001_layerMerge_slantRangeOffset_masked.tif',
             'fault_file':'/Users/chanagan/Downloads/Panama2026/GIS/ProfileTrace.shp',
             'map_swath':3000,
             'map_resolution': 80},
    }

fault = simple_trace.to_crs('EPSG:32617')

In [ ]:

import os
import numpy as np
import geopandas as gpd
import rasterio
import matplotlib.pyplot as plt

from shapely.geometry import LineString, MultiLineString, GeometryCollection
from shapely.ops import linemerge
from matplotlib.patches import Rectangle


# ============================================================
# Settings
# ============================================================

plot = True
sample_spacing = 80  # m

output_dir = (
    "/Users/chanagan/Downloads/Panama2026/"
    "NISAR_ASF/Profiles"
)
os.makedirs(output_dir, exist_ok=True)

plt.style.use("seaborn-v0_8-colorblind")


# ============================================================
# Helpers
# ============================================================

def get_fault_line(geom):
    """
    Return a LineString from a LineString, MultiLineString,
    or GeometryCollection.

    For disconnected traces, use the longest connected segment.
    """

    if geom is None or geom.is_empty:
        raise ValueError("Fault geometry is empty.")

    if geom.geom_type == "LineString":
        return geom

    if geom.geom_type == "MultiLineString":
        merged = linemerge(geom)

        if merged.geom_type == "LineString":
            return merged

        # Disconnected components: choose the longest component
        return max(merged.geoms, key=lambda g: g.length)

    if geom.geom_type == "GeometryCollection":
        lines = []

        for part in geom.geoms:
            if part.geom_type == "LineString":
                lines.append(part)
            elif part.geom_type == "MultiLineString":
                lines.extend(list(part.geoms))

        if not lines:
            raise ValueError(
                "GeometryCollection contains no line geometries."
            )

        merged = linemerge(lines)

        if merged.geom_type == "LineString":
            return merged

        return max(merged.geoms, key=lambda g: g.length)

    raise TypeError(
        f"Expected a line geometry, got {geom.geom_type}."
    )


def as_float_array(values):
    """Convert sampled values to float and convert masks to NaN."""

    if np.ma.isMaskedArray(values):
        return np.asarray(
            np.ma.filled(values, np.nan),
            dtype=float,
        )

    values = np.asarray(values, dtype=float)
    values[~np.isfinite(values)] = np.nan
    values[values == -9999] = np.nan

    return values


# ============================================================
# Process each swath
# ============================================================

for key, settings in swath_dict.items():

    print(f"\nProcessing: {key}")

    # --------------------------------------------------------
    # Read fault trace and raster
    # --------------------------------------------------------

    area_fault = gpd.read_file(
        settings["fault_file"]
    ).to_crs(crs)

    # Open raster once; do not overwrite the dataset variable
    with rasterio.open(settings["disp_file"]) as raster:

        line_geoms = area_fault.geometry.dropna()
        line_geoms = line_geoms[~line_geoms.is_empty]

        if line_geoms.empty:
            raise ValueError(f"No valid fault geometry for {key}")

        # Use the first feature, as in the original code
        line = get_fault_line(line_geoms.iloc[0])

        # ----------------------------------------------------
        # Sample points along the fault
        # ----------------------------------------------------

        d = np.arange(0, line.length, sample_spacing)

        if len(d) == 0 or d[-1] < line.length:
            d = np.append(d, line.length)

        pts = [line.interpolate(distance) for distance in d]

        xs = np.array([point.x for point in pts])
        ys = np.array([point.y for point in pts])

        # Calculate local fault azimuth using coordinate gradients
        dx = np.gradient(xs)
        dy = np.gradient(ys)

        az = (
            np.degrees(np.arctan2(dx, dy)) + 360
        ) % 360

        fault_points = gpd.GeoDataFrame(
            {"distance": d, "azimuth": az},
            geometry=pts,
            crs=area_fault.crs,
        )

        # ----------------------------------------------------
        # Initialize fit results
        # ----------------------------------------------------

        result_columns = [
            "par_intercept",
            "par_total_offset",
            "par_fault_loc",
            "par_shear_width",
            "par_total_offset_sig",
            "par_fault_loc_sig",
            "par_shear_width_sig",
            "par_slope1",
            "par_slope1_sig",
            "par_slope2",
            "par_slope2_sig",
            "par_fzw_offset",
            "par_ep_offset",
            "par_intercept_sig",
            "par_fit_sig",
        ]

        for col in result_columns:
            fault_points[col] = np.nan

        # ----------------------------------------------------
        # Fit profiles along the fault
        # ----------------------------------------------------

        for i in range(len(fault_points)):

            x0, y0 = xs[i], ys[i]
            azimuth = float(fault_points.azimuth.iloc[i])

            # Sample displacement profile.
            # Keep the raster dataset separate from sampled data.
            profile_values, spts, dists = tt.sample_swath(
                raster,
                x0,
                y0,
                azimuth,
                settings["swath_length"],
                settings["swath_width"],
                settings["resolution"],
                crs=crs,
            )

            # Ensure profile coordinates and values are usable
            profile_values = np.ma.filled(
                np.ma.asarray(profile_values), np.nan
            ).astype(float).squeeze()
            
            dists = np.asarray(dists).squeeze()
            
            # Both arrays should now be 1D
            if profile_values.ndim != 1 or dists.ndim != 1:
                raise ValueError(
                    f"Expected 1D profile arrays; got "
                    f"profile_values.shape={profile_values.shape}, "
                    f"dists.shape={dists.shape}"
                )
            
            if profile_values.size != dists.size:
                raise ValueError(
                    f"Profile length mismatch: {profile_values.size} values "
                    f"but {dists.size} distances"
                )
            
            # Remove invalid samples
            valid = np.isfinite(profile_values) & np.isfinite(dists)
            
            profile_values = profile_values[valid]
            dists_fit = dists[valid]
            
            # Avoid fitting profiles with insufficient valid samples
            if profile_values.size < 8:
                continue

            # Avoid fitting profiles with insufficient valid samples
            if len(profile_values) < 8:
                print(
                    f"  Skipping profile {i}: "
                    f"only {len(profile_values)} valid samples."
                )
                continue

            try:
                (
                    par_intercept,
                    par_total_offset,
                    par_fault_loc,
                    par_shear_width,
                    par_total_offset_sig,
                    par_fault_loc_sig,
                    par_shear_width_sig,
                    par_slope1,
                    par_slope1_sig,
                    par_slope2,
                    par_slope2_sig,
                    par_fzw_offset,
                    par_ep_offset,
                    par_intercept_sig,
                ) = tt.erf_curve_fit_twoslope(
                    profile_values,
                    dists_fit,
                )

                fitted = tt.erf_function_twoslope(
                    dists_fit,
                    par_intercept,
                    par_total_offset,
                    par_fault_loc,
                    par_shear_width,
                    par_slope1,
                    par_slope2,
                )

                par_fit_sig = np.nanstd(
                    profile_values - fitted
                )

            except (ValueError, RuntimeError, TypeError) as exc:
                print(f"  Fit failed for profile {i}: {exc}")
                continue

            results = {
                "par_intercept": par_intercept,
                "par_total_offset": par_total_offset,
                "par_fault_loc": par_fault_loc,
                "par_shear_width": par_shear_width,
                "par_total_offset_sig": par_total_offset_sig,
                "par_fault_loc_sig": par_fault_loc_sig,
                "par_shear_width_sig": par_shear_width_sig,
                "par_slope1": par_slope1,
                "par_slope1_sig": par_slope1_sig,
                "par_slope2": par_slope2,
                "par_slope2_sig": par_slope2_sig,
                "par_fzw_offset": par_fzw_offset,
                "par_ep_offset": par_ep_offset,
                "par_intercept_sig": par_intercept_sig,
                "par_fit_sig": par_fit_sig,
            }

            for col, value in results.items():
                fault_points.loc[i, col] = value

            # ------------------------------------------------
            # Plot profile and map
            # ------------------------------------------------

            if not plot:
                continue

            fig, axes = plt.subplots(
                1,
                2,
                figsize=(9, 3.5),
                gridspec_kw={"width_ratios": [1.8, 1]},
            )

            ax_prof, ax_par_map = axes

            # -----------------------------------------------
            # Profile plot
            # -----------------------------------------------

            ax_prof.scatter(
                dists_fit,
                profile_values,
                s=3,
                color="#1B9E77",
                label="Parallel",
            )

            ax_prof.plot(
                dists_fit,
                fitted,
                color="#0F5943",
                lw=1.2,
                label="Two-slope ERF fit",
            )

            ax_prof.axvline(
                par_fault_loc,
                color="#0F5943",
                ls="--",
                lw=1,
            )

            ax_prof.axvspan(
                par_fault_loc - 2 * par_shear_width,
                par_fault_loc + 2 * par_shear_width,
                color="#0F5943",
                alpha=0.15,
            )

            bbox_props = {
                "facecolor": "lightgrey",
                "alpha": 0.5,
                "edgecolor": "none",
            }

            ax_prof.text(
                0.02,
                0.98,
                f"Offset: {par_total_offset:.2f}"
                f" ± {4 * par_total_offset_sig:.2f}\n"
                f"Width: {par_shear_width:.0f}"
                f" ± {4 * par_shear_width_sig:.0f} m",
                transform=ax_prof.transAxes,
                va="top",
                fontsize=7,
                color="#0F5943",
                bbox=bbox_props,
            )

            ax_prof.set_xlabel("Distance from profile center (m)")
            ax_prof.set_ylabel("Displacement / velocity")
            ax_prof.legend(fontsize=7, loc="lower left")

            # -----------------------------------------------
            # Map swath sampling
            # -----------------------------------------------

            map_values, map_pts, map_dists = tt.sample_swath(
                raster,
                x0,
                y0,
                azimuth,
                settings["swath_length"] * 1.5,
                settings["map_swath"],
                settings["map_resolution"],
                crs=crs,
            )

            map_values = as_float_array(map_values)

            # Normalize shapes
            map_pts = np.asarray(map_pts).reshape(-1, 2)

            map_values = np.ma.filled(
                np.ma.asarray(map_values), np.nan
            ).astype(float).reshape(-1)

            print("map_pts:", map_pts.shape)
            print("map_values:", map_values.shape)
            
            if map_pts.shape[0] != map_values.size:
                raise ValueError(
                    f"Mismatch: {map_pts.shape[0]} points, "
                    f"{map_values.size} values"
                )
            
            valid_map = (
                np.isfinite(map_values)
                & np.isfinite(map_pts[:, 0])
                & np.isfinite(map_pts[:, 1])
            )
            
            print("valid_map:", valid_map.shape)
            
            sc = ax_par_map.scatter(
                map_pts[valid_map, 0],
                map_pts[valid_map, 1],
                c=map_values[valid_map],
                s=5,
                cmap="RdBu_r",
                vmin=-4,
                vmax=4,
            )

            area_fault.plot(
                ax=ax_par_map,
                color="black",
                linewidth=0.8,
            )

            # Plot fitted fault location
            rad = np.deg2rad(azimuth)

            # Perpendicular direction to the fault
            u_perp = np.array([
                np.cos(rad),
                -np.sin(rad),
            ])

            fault_xy = (
                np.array([x0, y0])
                + u_perp * par_fault_loc
            )

            ax_par_map.plot(
                fault_xy[0],
                fault_xy[1],
                "r+",
                ms=8,
                mew=1.5,
            )

            # -----------------------------------------------
            # Swath footprint
            # -----------------------------------------------

            # Profile is perpendicular to the fault azimuth.
            # Compute the four corners explicitly so the
            # footprint is correctly rotated in map coordinates.
            u_along = np.array([
                np.sin(rad),
                np.cos(rad),
            ])

            half_length = settings["swath_length"] / 2
            half_width = settings["swath_width"] / 2

            center = np.array([x0, y0])

            corners = np.array([
                center - half_length * u_perp - half_width * u_along,
                center + half_length * u_perp - half_width * u_along,
                center + half_length * u_perp + half_width * u_along,
                center - half_length * u_perp + half_width * u_along,
            ])

            ax_par_map.add_patch(
                plt.Polygon(
                    corners,
                    closed=True,
                    fill=False,
                    edgecolor="black",
                    linewidth=0.7,
                )
            )

            pad = settings["map_swath"]

            ax_par_map.set_xlim(x0 - pad, x0 + pad)
            ax_par_map.set_ylim(y0 - pad, y0 + pad)
            ax_par_map.set_aspect("equal")
            ax_par_map.set_xticks([])
            ax_par_map.set_yticks([])
            ax_par_map.set_title("Parallel", fontsize=9)

            fig.colorbar(
                sc,
                ax=ax_par_map,
                fraction=0.046,
                pad=0.03,
                label="Displacement / velocity",
            )

            fig.suptitle(
                f"{key} | Profile {i}",
                fontsize=9,
            )

            plt.tight_layout()

            out_plot = os.path.join(
                output_dir,
                f"ErfProfile{i}_{key}"
                f"_ProfLen{settings['swath_length']}"
                f"_SwathWidth{settings['swath_width']}"
                f"_every{settings['resolution']}m.pdf",
            )

            fig.savefig(
                out_plot,
                dpi=300,
                bbox_inches="tight",
            )

            plt.show()
            plt.close(fig)

    # ========================================================
    # Save results for this fault
    # ========================================================

    out_csv = os.path.join(
        output_dir,
        f"ErfProfileResults_{key}"
        f"_ProfLen{settings['swath_length']}"
        f"_SwathWidth{settings['swath_width']}"
        f"_every{settings['resolution']}m_unfiltered.csv",
    )

    fault_points.to_csv(
        out_csv,
        index=False,
        float_format="%.4f",
    )

    print(f"Saved results: {out_csv}")


In [ ]:
17000/80

In [ ]:
map_pts

In [ ]:
# Re-save erf with bad values out
import pandas as pd
import geopandas as gpd
from shapely import wkt
from shapely.geometry import LineString, Point
import numpy as np

files = ['/Users/chanagan/Downloads/Panama2026/NISAR_ASF/Profiles/ErfProfileResults_Panama_ProfLen10000_SwathWidth80_every80m_unfiltered.csv']

names = [
    "Panama2026"
]

crs = "EPSG:32617"

for name, file in zip(names, files):

    fault_points = pd.read_csv(file)
    fault_points = fault_points[fault_points.par_shear_width < 5000]
    fault_points = fault_points[fault_points.par_total_offset > 0]

    fault_points["geometry"] = fault_points["geometry"].apply(wkt.loads)

    fault_points = gpd.GeoDataFrame(
        fault_points,
        geometry="geometry",
        crs=crs,
    )

    # ----------------------------------------------------
    # Build shifted profile lines
    # ----------------------------------------------------
    perp_lines = []
    shifted_points = []

    for _, row in fault_points.iterrows():

        x = row.geometry.x
        y = row.geometry.y

        # swath azimuth (perpendicular to fault)
        swath_az = (row.azimuth + 90) % 360
        theta = np.deg2rad(swath_az)

        # Shift profile center to fitted fault location
        shift = row.par_fault_loc

        x_shift = x + np.sin(theta) * shift
        y_shift = y + np.cos(theta) * shift

        shifted_points.append(Point(x_shift, y_shift))

        # Build profile centered on shifted location
        half = row.par_shear_width * 2
        dx = np.sin(theta) * half
        dy = np.cos(theta) * half

        perp_lines.append(
            LineString(
                [
                    (x_shift - dx, y_shift - dy),
                    (x_shift + dx, y_shift + dy),
                ]
            )
        )

    # ----------------------------------------------------
    # Export profile lines
    # ----------------------------------------------------
    out = gpd.GeoDataFrame(
        {
            "displacement_magnitude_m": np.round(fault_points["par_total_offset"],4),
            "displacement_uncertainty_m": np.round(fault_points["par_fit_sig"],4),
            "deformation_zone_width_m": np.round(fault_points["par_shear_width"],4),
            "geometry": shifted_points,
            "geometry": perp_lines,
        },
        crs=crs,
    )

    out.to_file(
        #f"/Volumes/Venezuela/Venezuela2026Doublet/WorldViewBasic/MeasurementsFiles/{name}_Profiles.geojson",
        f'/Users/chanagan/Downloads/Panama2026/NISAR_ASF/Profiles/{name}_Profiles.geojson',
        driver="GeoJSON",
    )
    

    # ----------------------------------------------------
    # Export shifted centers
    # ----------------------------------------------------
    centers = gpd.GeoDataFrame(
        {
            "displacement_magnitude_m": np.round(fault_points["par_total_offset"],4),
            "displacement_uncertainty_m": np.round(fault_points["par_fit_sig"],4),
            "deformation_zone_width_m": np.round(fault_points["par_shear_width"],4),
            "geometry": shifted_points,
        },
        crs=crs,
    )

    #centers.to_file(
    #    f"/Volumes/Venezuela/Venezuela2026Doublet/WorldViewBasic/MeasurementsFiles/{name}_ShiftedCenters.geojson",
    #    driver="GeoJSON",
    #)

In [ ]:
centers.deformation_zone_width_m.mean()

In [ ]:
from shapely.ops import linemerge
import matplotlib.pyplot as plt
plot = True

for key in list(swath_dict.keys()):
    print(key)
    # Read the rasters
    disp = rasterio.open(swath_dict[key]['disp_file'],masked=True)
    area_fault = gpd.read_file(swath_dict[key]['fault_file']).to_crs(crs)
    line = area_fault.geometry.iloc[0]

    # Sample every 50 m
    d = np.arange(0, line.length, 80)
    pts = [line.interpolate(x) for x in d]

    xs = np.array([p.x for p in pts])
    ys = np.array([p.y for p in pts])

    # Central differences (forward/backward at ends)
    dx = np.gradient(xs)
    dy = np.gradient(ys)

    az = (np.degrees(np.arctan2(dx, dy)) + 360) % 360

    fault_points = gpd.GeoDataFrame(
        {"distance": d, "azimuth": az},
        geometry=pts,
        crs=area_fault.crs,
    )

    for i in range(len(xs)):
        x0, y0 = xs[i], ys[i]
    
        disp, spts, dists = tt.sample_swath(
            disp, x0, y0,
            fault_points.azimuth.iloc[i], swath_dict[key]['swath_length'], 
            swath_dict[key]['swath_width'], swath_dict[key]['resolution'],
            crs=crs
        )
        disp[disp == -9999] = np.nan
        (
            par_intercept,
            par_total_offset,
            par_fault_loc,
            par_shear_width,
            par_total_offset_sig,
            par_fault_loc_sig,
            par_shear_width_sig,
            par_slope1,
            par_slope1_sig,
            par_slope2,
            par_slope2_sig,
            par_fzw_offset,
            par_ep_offset,
            par_intercept_sig,
        ) = tt.erf_curve_fit_twoslope(disp, dists)
    
        par_fit_sig = np.nanstd(
            disp
            - tt.erf_function_twoslope(
                dists,
                par_intercept,
                par_total_offset,
                par_fault_loc,
                par_shear_width,
                par_slope1,
                par_slope2,
            )
        )
    
        fault_points.loc[i, "par_intercept"] = par_intercept
        fault_points.loc[i, "par_total_offset"] = par_total_offset
        fault_points.loc[i, "par_fault_loc"] = par_fault_loc
        fault_points.loc[i, "par_shear_width"] = par_shear_width
        fault_points.loc[i, "par_total_offset_sig"] = par_total_offset_sig
        fault_points.loc[i, "par_fault_loc_sig"] = par_fault_loc_sig
        fault_points.loc[i, "par_shear_width_sig"] = par_shear_width_sig
        fault_points.loc[i, "par_slope1"] = par_slope1
        fault_points.loc[i, "par_slope1_sig"] = par_slope1_sig
        fault_points.loc[i, "par_slope2"] = par_slope2
        fault_points.loc[i, "par_slope2_sig"] = par_slope2_sig
        fault_points.loc[i, "par_fzw_offset"] = par_fzw_offset
        fault_points.loc[i, "par_ep_offset"] = par_ep_offset
        fault_points.loc[i, "par_fit_sig"] = par_fit_sig
    
        if plot == True:
            # ===============================
            # PLOTTING (WITH FAULT TRACE + INTERSECTIONS)
            # ===============================    
            plt.style.use('seaborn-v0_8-colorblind')

            fig, axes = plt.subplots(
                1, 3,
                figsize=(10, 3),
                gridspec_kw={'width_ratios': [2.5, 1, 1]}
            )

            ax_prof, ax_par_map = axes

            # =========================
            # 1) PROFILE GEOMETRY
            # =========================

            x0, y0 = xs[i], ys[i]
            rad = np.deg2rad(fault_points.azimuth.iloc[i])

            # unit vector perpendicular to fault (profile direction)
            u_perp = np.array([np.cos(rad), -np.sin(rad)])

            # profile endpoints
            p1 = np.array([x0, y0]) - u_perp * (swath_dict[key]['swath_length'] / 2)
            p2 = np.array([x0, y0]) + u_perp * (swath_dict[key]['swath_length'] / 2)

            # =========================
            # 3) PROFILE PLOT
            # =========================
            # data
            ax_prof.scatter(dists, par, s=2, c='#1B9E77', label='Parallel')

            # fits
            ax_prof.plot(
                dists,
                tt.erf_function_twoslope(
                    dists,
                    par_intercept, par_total_offset, par_fault_loc,
                    par_shear_width, par_slope1, par_slope2
                ),
                color="#0F5943", lw=1.5
            )
            # ERF fault (center)
            ax_prof.axvline(par_fault_loc, color="#0F5943", ls='--', lw=1)# label='ERF fault')
            # shear zone shading (parallel)
            ax_prof.axvspan(
                par_fault_loc-2*par_shear_width,
                par_fault_loc+2*par_shear_width,
                color="#0F5943", alpha=0.15
            )

            bbox_props = dict(facecolor='lightgrey', alpha=0.5, edgecolor='none')

            ax_prof.text(
                0.02, 0.98,
                f'Par: {par_total_offset:.2f}±{4*par_total_offset_sig:.2f} cm/yr\n'
                f'W: {par_shear_width:.0f}±{4*par_shear_width_sig:.0f} m',
                transform=ax_prof.transAxes,
                va='top',
                fontsize=7,
                color="#0F5943",
                bbox=bbox_props
                )
            
            # labels
            ax_prof.set_xlabel('Distance from fault (m)')
            ax_prof.set_ylabel('Velocity (cm/yr)')
            ax_prof.legend(fontsize=7, loc='lower left')

            # =========================
            # 4) MAP DATA
            # =========================

            disp_map_samps, map_pts, map_dists = tt.sample_swath(
                disp, x0, y0,
                fault_points.azimuth.iloc[i],
                swath_dict[key]['swath_length']*1.5, swath_dict[key]['map_swath'], 
                swath_dict[key]['map_resolution'], crs=crs
            )
            disp_map_samps[disp_map_samps == -9999] = np.nan

            # =========================
            # 5) MAP PLOTS
            # =========================

            # parallel
            ax_par_map.scatter(
                map_pts[:,0], map_pts[:,1],
                c=disp_map_samps, s=5, cmap='RdBu_r',
                vmin=-4, vmax=4
            )

            # fault trace
            area_fault.plot(ax=ax_par_map, color='k', linewidth=1)

            ax_par_map.set_title('Parallel', fontsize=9)

            # =========================
            # 6) SWATH BOX + LIMITS
            # =========================

            xll, yll = swath_corners(
                x0, y0,
                fault_points.azimuth.iloc[i],
                swath_dict[key]['swath_length'], swath_dict[key]['swath_width']
            )[-1]

            pad = swath_dict[key]['map_swath']

            for ax in [ax_par_map]:
                ax.set_xlim(x0 - pad, x0 + pad)
                ax.set_ylim(y0 - pad, y0 + pad)

                ax.add_patch(Rectangle(
                    [xll, yll],
                     swath_dict[key]['swath_length'], swath_dict[key]['swath_width'],
                    angle=-fault_points.azimuth.iloc[i],
                    facecolor='none',
                    edgecolor='black',
                    linewidth=0.5,
                    alpha=0.5
                ))

                ax.set_aspect('equal')
                ax.set_xticks([])
                ax.set_yticks([])

            # =========================
            # 7) ERF FAULT LOCATION ON MAP
            # =========================

            fault_xy = np.array([x0, y0]) + u_perp * par_fault_loc

            ax_par_map.plot(fault_xy[0], fault_xy[1], 'r+', ms=8, mew=1.5)

            fig.suptitle(f'ErfProfile{i}_{key}_ProfLen{swath_dict[key]['swath_length']}_SwathWidth{ swath_dict[key]['swath_width']}_every{swath_dict[key]['resolution']}m.pdf')

            # nudge map panels slightly right
            pos_par = ax_par_map.get_position()

            shift = 0.02  # adjust if needed

            ax_par_map.set_position([pos_par.x0 + shift, pos_par.y0, pos_par.width, pos_par.height])
            # tighten spacing between the two map panels
            gap_reduce = 0.015

            # =========================
            plt.tight_layout()
            plt.savefig(
                    f'/Users/chanagan/Downloads/Panama2026/NISAR_ASF/Profiles/ErfProfile{i}'
                    f'_{key}_ProfLen{swath_dict[key]['swath_length']}_SwathWidth{swath_dict[key]['swath_width']}_every{swath_dict[key]['resolution']}m.pdf',
                    dpi=300
                )
            plt.show()
    fault_points.to_csv(
        f'/Users/chanagan/Downloads/Panama2026/NISAR_ASF/Profiles/ErfProfileResults_{key}_ProfLen{swath_dict[key]['swath_length']}_SwathWidth{ swath_dict[key]['swath_width']}_every{swath_dict[key]['resolution']}m_unfiltered.csv',
        float_format='%.4f'
    )

### C11 correlation processing


In [ ]:
import os
import sys
sys.path.append('/Users/chanagan/Documents/GitHub/ImageryResources/Functions/')
import TiffTools as tt

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

from pyproj import Proj, Geod
import rasterio

%load_ext autoreload
%autoreload 2

In [ ]:
folder = '/Users/chanagan/Downloads/Panama2026/MEC_C11_noSpeckleFilt/'
prefiles = ['/Users/chanagan/Downloads/Panama2026/MEC_C11_speckleFilt/C11_20260927_mosaic_EPSG32617.tif',
            '/Users/chanagan/Downloads/Panama2026/MEC_C11_speckleFilt/C11_20261009_mosaic_EPSG32617.tif']

tt.micmacPostProcessing(folder=folder,
                             prefiles=prefiles,
                             outprefix=folder)

In [ ]:
angle = 288

with rasterio.open(f'{folder}Correlmicmac.tif') as csrc:
        corr = csrc.read(1)
        mask = ~(corr < 0.3)

par, perp = tt.projectDisp(f'{folder}EWmicmac.tif',
                            f'{folder}NSmicmac.tif',
                            angle,
                            mask=mask,
                            partif=f'{folder}DispParallelToward{angle}.tif',
                            perptif=f'{folder}DispPerpendicularToward{angle}.tif')
files = [f'{folder}EWmicmac.tif',
        f'{folder}NSmicmac.tif']
for file in files:
    with rasterio.open(file) as src:
        arr = src.read(1)
        profile = src.profile
        nodata = src.nodata
    with rasterio.open(f'{folder}Correlmicmac.tif') as csrc:
        corr = csrc.read(1)
        mask = (corr < 0.2)

    arr[mask] = -9999
    with rasterio.open(f"{file[:-4]}_masked.tif", "w", **profile) as dst:
            dst.write(arr, 1)

In [ ]:
mask

In [ ]:
with rasterio.open(f'{folder}Correlmicmac.tif') as csrc:
        corr = csrc.read(1)
        mask = (corr < 0.2)

folder = '/Users/chanagan/Downloads/Panama2026/MEC_C11/'
prefiles = ['/Users/chanagan/Downloads/Panama2026/MEC_C11/C11_20260927_mosaic_EPSG32617.tif',
            '/Users/chanagan/Downloads/Panama2026/MEC_C11/C11_20261009_mosaic_EPSG32617.tif']

tt.micmacPostProcessing(folder=folder,
                             prefiles=prefiles,
                             outprefix=folder)

import rasterio
files = [f'{folder}NSmicmac.tif',
         f'{folder}EWmicmac.tif']

#landmask_file = '/Volumes/Venezuela/Venezuela2026Doublet/S2_GEE/gshhg-shp-2.3.7/GSHHS_shp/f/GSHHS_f_L1_epsg32619.shp'

for file in files:
    with rasterio.open(file) as src:
        arr = src.read(1)
        profile = src.profile
        nodata = src.nodata
    with rasterio.open(f'{folder}Correlmicmac.tif') as csrc:
        corr = csrc.read(1)
        mask = (corr < 0.2)
    arr[mask] = -9999
    with rasterio.open(f"{file[:-4]}_{folder.split('/')[-2][4:]}_masked.tif", "w", **profile) as dst:
            dst.write(arr, 1)

In [ ]:
for file in files:
    with rasterio.open(file) as src:
        arr = src.read(1)
        profile = src.profile
        nodata = src.nodata
    with rasterio.open(f'{folder}Correlmicmac.tif') as csrc:
        corr = csrc.read(1)
        mask = (corr < 0.2)
    arr[mask] = -9999
    with rasterio.open(f"{file[:-4]}_{folder.split('/')[-2][4:]}_masked.tif", "w", **profile) as dst:
            dst.write(arr, 1)